# Project M3S - Pre-training 105M Swarm Transformer from Scratch (V2 Fix)
- 100% Zero <UNK> BPE Tokenizer
- LLaMA Architecture: 105M Parameters
- Free T4 GPU (~15 minutes for 5 epochs)

In [ ]:
# 1. Install Dependencies
!pip install torch transformers datasets accelerate onnx tokenizers

In [ ]:
# 2. Train Custom BPE Tokenizer (Zero UNK)
import json
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast

raw_tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
raw_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
raw_tokenizer.decoder = decoders.ByteLevel()

tok_pad = "<" + "PAD>"
tok_unk = "<" + "UNK>"
tok_bos = "<" + "BOS>"
tok_eos = "<" + "EOS>"
tok_list = [tok_pad, tok_unk, tok_bos, tok_eos]

trainer_bpe = trainers.BpeTrainer(vocab_size=256, special_tokens=tok_list, show_progress=True)

with open("m3s_105m_pretrain.jsonl", "r", encoding="utf-8") as f:
    texts = [json.loads(line)["text"] for line in f]

raw_tokenizer.train_from_iterator(texts, trainer_bpe)
raw_tokenizer.save("m3s_tokenizer.json")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="m3s_tokenizer.json",
    bos_token=tok_bos,
    eos_token=tok_eos,
    unk_token=tok_unk,
    pad_token=tok_pad
)
print("BPE Tokenizer Ready! Vocab Size:", tokenizer.vocab_size)
print("Sample:", tokenizer.convert_ids_to_tokens(tokenizer.encode(texts[0])[:12]))

In [ ]:
# 3. Instantiate 105M Llama Model
from transformers import LlamaConfig, LlamaForCausalLM

config = LlamaConfig(
    vocab_size = 256,
    hidden_size = 768,
    intermediate_size = 2048,
    num_hidden_layers = 12,
    num_attention_heads = 12,
    max_position_embeddings = 512,
    rms_norm_eps = 1e-6,
    initializer_range = 0.02,
    use_cache = True,
    pad_token_id = 0,
    bos_token_id = 2,
    eos_token_id = 3
)
model = LlamaForCausalLM(config)
print("Param Count:", sum(p.numel() for p in model.parameters()))

In [ ]:
# 4. Tokenize Dataset
from datasets import load_dataset
dataset = load_dataset("json", data_files="m3s_105m_pretrain.jsonl", split="train")
tokenized_dataset = dataset.map(lambda ex: {"input_ids": tokenizer.encode(ex["text"])}, remove_columns=["text"])
print("Tokenized Sample:", tokenized_dataset[0]["input_ids"][:15])

In [ ]:
# 5. Pre-train Model
import torch
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir = "./m3s_105m_checkpoints",
    num_train_epochs = 5,
    per_device_train_batch_size = 32,
    gradient_accumulation_steps = 2,
    learning_rate = 5e-4,
    warmup_steps = 100,
    weight_decay = 0.01,
    logging_steps = 25,
    save_steps = 500,
    fp16 = torch.cuda.is_available(),
    dataloader_num_workers = 0,
    report_to = "none"
)

def custom_collate_fn(batch):
    input_ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=0)
    labels = padded.clone()
    labels[labels == 0] = -100
    return {"input_ids": padded, "labels": labels}

trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_dataset, data_collator=custom_collate_fn)
trainer.train()
print("Pre-training Complete!")

In [ ]:
# 6. Test Inference
model.eval()
test_prompt = "<BOS>SIT:M1_ORE_DIAMOND_COUNT16_Y-58_HP_FULL:M2_IDLE:C1_IDLE"
inp_ids = torch.tensor([tokenizer.encode(test_prompt)]).to(model.device)
with torch.no_grad():
    out = model.generate(inp_ids, max_new_tokens=40, eos_token_id=3)
print("Inference Output:", tokenizer.decode(out[0]))

In [ ]:
# 7. Save & Zip Model
import shutil
model.save_pretrained("m3s_105m_final")
tokenizer.save_pretrained("m3s_105m_final")
shutil.make_archive("/content/m3s_105m_final", "zip", "m3s_105m_final")
print("ZIP Created at /content/m3s_105m_final.zip")